# Scene Detection



In [ ]:
from collections.abc import Mapping, Sequence
from typing import Any

import pandas as pd
import numpy as np
import cv2
from PIL import Image

from pathlib import Path
from genericpath import exists

from tqdm.auto import tqdm

from scipy.sparse import csr_matrix, diags
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
!pip install git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git

  Cloning https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git to /tmp/pip-req-build-r203g5_r
  Running command git clone --filter=blob:none --quiet https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git /tmp/pip-req-build-r203g5_r
  Resolved https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git to commit 23ad6fb41b296fb9258b0e7825125a914573b906
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for omnishotcut: filename=omnishotcut-0.1.0-py3-none-any.whl size=28515 sha256=0b4b46df1bf49332b45b5e13429c8b92b6ed421c78728ba91a8f750a23b7585e
  Stored in directory: /tmp/pip-ephem-wheel-cache-obsvnmsd/wheels/38/73/d1/f7ba58b26580940b3b89d63a82b767e340d3a3e48e2e139068
Successfully built omnishotcut


In [ ]:
BBC_ROOT = Path("/content/bbc")
bbc_movies = [f"bbc_{n:02d}" for n in range(1, 12)]

FOLDER_ID = "1SbqWxfZ6YHBNAIFwtraJZXl1K3qNq3M_"


def _assets_present():
    gt_ok = all(
        (BBC_ROOT / "GT" / f"{m}_shots.csv").exists()
        and (BBC_ROOT / "GT" / f"{m}_scenes.csv").exists()
        for m in bbc_movies
    )
    videos_ok = all((BBC_ROOT / f"{m}.mp4").exists() for m in bbc_movies)
    dino_ok = (BBC_ROOT / "models" / "dinov3-vitb16" / "config.json").exists()
    return gt_ok and videos_ok and dino_ok


if not _assets_present():
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    auth.authenticate_user()
    _drive = build("drive", "v3")

    def _list_children(folder_id):
        items, page_token = [], None
        while True:
            resp = (
                _drive.files()
                .list(
                    q=f"'{folder_id}' in parents and trashed=false",
                    fields="nextPageToken, files(id, name, mimeType)",
                    pageSize=1000,
                    pageToken=page_token,
                    supportsAllDrives=True,
                    includeItemsFromAllDrives=True,
                )
                .execute()
            )
            items.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                return items

    def _download_tree(folder_id, dest):
        dest = Path(dest)
        dest.mkdir(parents=True, exist_ok=True)
        for item in _list_children(folder_id):
            target = dest / item["name"]
            if item["mimeType"] == "application/vnd.google-apps.folder":
                _download_tree(item["id"], target)
            elif not target.exists():
                request = _drive.files().get_media(fileId=item["id"], supportsAllDrives=True)
                with open(target, "wb") as fh:
                    downloader = MediaIoBaseDownload(fh, request, chunksize=64 * 1024 * 1024)
                    done = False
                    while not done:
                        _, done = downloader.next_chunk()
                print(f"  {target.relative_to(BBC_ROOT)}")

    print("Downloading assets from the shared Drive folder to local Colab (Drive API)...")
    _download_tree(FOLDER_ID, BBC_ROOT)

print("BBC_ROOT =", BBC_ROOT, "| assets present:", _assets_present())

  models/dinov3-vitb16/model.safetensors
  models/dinov3-vitb16/config.json
  models/dinov3-vitb16/preprocessor_config.json
  omnishot/bbc_11_establishing.npy
  omnishot/bbc_10_establishing.npy
  omnishot/bbc_09_establishing.npy
  omnishot/bbc_08_establishing.npy
  omnishot/bbc_07_establishing.npy
  omnishot/bbc_06_establishing.npy
  omnishot/bbc_05_establishing.npy
  omnishot/bbc_04_establishing.npy
  omnishot/bbc_03_establishing.npy
  omnishot/bbc_02_establishing.npy
  omnishot/bbc_01_establishing.npy
  omnishot/bbc_11_all_vectors.npy
  omnishot/bbc_10_all_vectors.npy
  omnishot/bbc_09_all_vectors.npy
  omnishot/bbc_08_all_vectors.npy
  omnishot/bbc_07_all_vectors.npy
  omnishot/bbc_06_all_vectors.npy
  omnishot/bbc_05_all_vectors.npy
  omnishot/bbc_04_all_vectors.npy
  omnishot/bbc_03_all_vectors.npy
  omnishot/bbc_02_all_vectors.npy
  omnishot/bbc_01_all_vectors.npy
  omnishot/bbc_11_clean_vectors.npy
  omnishot/bbc_10_clean_vectors.npy
  omnishot/bbc_09_clean_vectors.npy
  omnisho

# Read Ground Truth

In [ ]:
shots_frames = []
for movie in bbc_movies:
    movie_shots = pd.read_csv(
        BBC_ROOT / "GT" / f"{movie}_shots.csv",
        header=0,
        names=["start_frame", "end_frame"],
    )
    movie_shots.insert(0, "movie_name", movie)
    shots_frames.append(movie_shots)

bbc_shots_gt = pd.concat(shots_frames, ignore_index=True)
bbc_shots_gt["shot_number"] = bbc_shots_gt.groupby("movie_name").cumcount() + 1

In [ ]:
bbc_shots_gt[bbc_shots_gt["movie_name"] == "bbc_07"].head(50)

,movie_name,start_frame,end_frame,shot_number
2733,bbc_07,1,650,1
2734,bbc_07,651,818,2
2735,bbc_07,819,895,3
2736,bbc_07,896,947,4
2737,bbc_07,948,984,5
2738,bbc_07,985,1026,6
2739,bbc_07,1027,1100,7
2740,bbc_07,1101,1123,8
2741,bbc_07,1124,1145,9
2742,bbc_07,1146,1233,10


In [ ]:
scenes_frames = []
for movie in bbc_movies:
    movie_scenes = pd.read_csv(
        BBC_ROOT / "GT" / f"{movie}_scenes.csv",
        header=0,
        names=["start_shot", "end_shot"],
    )
    movie_scenes.insert(0, "movie_name", movie)
    scenes_frames.append(movie_scenes)

bbc_scenes_gt = pd.concat(scenes_frames, ignore_index=True)

bbc_scenes_gt = bbc_scenes_gt.merge(
    bbc_shots_gt[
        ["movie_name", "shot_number", "start_frame"]
    ].rename(
        columns={
            "shot_number": "start_shot",
            "start_frame": "start_frame",
        }
    ),
    on=["movie_name", "start_shot"],
    how="left",
)

bbc_scenes_gt = bbc_scenes_gt.merge(
    bbc_shots_gt[
        ["movie_name", "shot_number", "end_frame"]
    ].rename(
        columns={
            "shot_number": "end_shot",
            "end_frame": "end_frame",
        }
    ),
    on=["movie_name", "end_shot"],
    how="left",
)

In [ ]:
bbc_scenes_gt[bbc_scenes_gt["movie_name"] == "bbc_07"].head(50)

,movie_name,start_shot,end_shot,start_frame,end_frame
382,bbc_07,1,1,1,650
383,bbc_07,2,14,651,1981
384,bbc_07,15,24,1982,3533
385,bbc_07,25,31,3534,4850
386,bbc_07,32,35,4851,6609
387,bbc_07,36,41,6610,8176
388,bbc_07,42,45,8177,8921
389,bbc_07,46,53,8922,9431
390,bbc_07,54,55,9432,9781
391,bbc_07,56,60,9782,10407


# Generate Shots

In [ ]:
video_path = BBC_ROOT / "bbc_01.mp4"

if not video_path.exists():
    raise ValueError(f"Video file not found: {video_path}")

In [ ]:
import omnishotcut

cut_model = omnishotcut.load("uva-cv-lab/OmniShotCut", filename = "OmniShotCut_ckpt.pth")

INFO:omnishotcut:Downloading checkpoint from HuggingFace: uva-cv-lab/OmniShotCut ...


OmniShotCut_ckpt.pth: reconstructing file:   0%|          |  0.00B /  164MB            

OmniShotCut_ckpt.pth: downloading bytes:           |  0.00B            

Loading OmniShotCut from /root/.cache/huggingface/hub/models--uva-cv-lab--OmniShotCut/snapshots/7f646c4ff4bb843e18c013481fb5d9ed2b068c6b/OmniShotCut_ckpt.pth ...
INFO:omnishotcut:Loading OmniShotCut from /root/.cache/huggingface/hub/models--uva-cv-lab--OmniShotCut/snapshots/7f646c4ff4bb843e18c013481fb5d9ed2b068c6b/OmniShotCut_ckpt.pth ...


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 244MB/s]
OmniShotCut loaded successfully.
INFO:omnishotcut:OmniShotCut loaded successfully.


In [ ]:
OMNI_DIR = BBC_ROOT / "omnishot"
OMNI_DIR.mkdir(parents=True, exist_ok=True)


def generate_or_load_omni_shots(movie):
    cache = OMNI_DIR / f"{movie}_omni_shots.csv"
    if cache.exists():
        return pd.read_csv(cache)

    movie_video = BBC_ROOT / f"{movie}.mp4"
    if not movie_video.exists():
        raise FileNotFoundError(f"Video not found: {movie_video}")

    ranges, intra_labels, inter_labels = cut_model.inference(
        str(movie_video), mode="default"
    )

    df = pd.DataFrame(
        {
            "movie_name": movie,
            "shot_index": np.arange(len(ranges)),
            "start_frame": [r[0] for r in ranges],
            "end_frame": [r[1] for r in ranges],
            "intra_label": intra_labels,
            "inter_label": inter_labels,
        }
    )
    df["duration_frames"] = df["end_frame"] - df["start_frame"] + 1

    df.to_csv(cache, index=False)
    return df


omni_shots_all = pd.concat(
    [generate_or_load_omni_shots(movie) for movie in tqdm(bbc_movies)],
    ignore_index=True,
)

  0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
omni_shots_all[omni_shots_all["movie_name"] == "bbc_05"].head(25)

,movie_name,shot_index,start_frame,end_frame,intra_label,inter_label,duration_frames
1785,bbc_05,0,0,27,General,New_Start,28
1786,bbc_05,1,27,650,General,New_Start,624
1787,bbc_05,2,650,1632,General,New_Start,983
1788,bbc_05,3,1632,1908,General,New_Start,277
1789,bbc_05,4,1908,2085,General,New_Start,178
1790,bbc_05,5,2085,2188,General,New_Start,104
1791,bbc_05,6,2188,2336,General,New_Start,149
1792,bbc_05,7,2336,2532,General,New_Start,197
1793,bbc_05,8,2532,2740,General,New_Start,209
1794,bbc_05,9,2740,2900,General,New_Start,161


# Create Embeddings

In [ ]:
FRACTIONS = (0.25, 0.50, 0.75)

In [ ]:
class FrameReader:
    def __init__(self, video_path):
        self.video_path = video_path
        self.cap = cv2.VideoCapture(video_path)

        if not self.cap.isOpened():
            raise ValueError(f"Could not open video: {video_path}")

    def read(self, frame_idx):
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame_bgr = self.cap.read()

        if not ok:
            raise ValueError(f"Could not read frame {frame_idx}")

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        return Image.fromarray(frame_rgb)

    def close(self):
        self.cap.release()

In [ ]:
def sample_interior_frame_indices(start_frame, end_frame, fractions=FRACTIONS, end_inclusive=False):
    """
    Returns frame indices at fractions through a shot.

    If OmniShotCut's end_frame is inclusive, keep end_inclusive=True.
    If end_frame is exclusive, set end_inclusive=False.
    """
    start_frame = int(start_frame)
    end_frame = int(end_frame)

    last_frame = end_frame if end_inclusive else end_frame - 1
    last_frame = max(last_frame, start_frame)

    span = last_frame - start_frame

    return [
        int(round(start_frame + frac * span))
        for frac in fractions
    ]

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel

model_id = "facebook/dinov3-vitb16-pretrain-lvd1689m"
DINOV3_DIR = BBC_ROOT / "models" / "dinov3-vitb16"   # persists on Drive

device = "cuda" if torch.cuda.is_available() else "cpu"

if not (DINOV3_DIR / "config.json").exists():
    # First time: download from HF (needs your HF login for this gated repo), save to Drive.
    DINOV3_DIR.mkdir(parents=True, exist_ok=True)
    AutoImageProcessor.from_pretrained(model_id).save_pretrained(DINOV3_DIR)
    AutoModel.from_pretrained(model_id).save_pretrained(DINOV3_DIR)

processor = AutoImageProcessor.from_pretrained(DINOV3_DIR)
dino_model = AutoModel.from_pretrained(DINOV3_DIR).to(device)
dino_model.eval()

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (model): DINOv3ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x DINOv3ViTLayer(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attention): DINOv3ViTAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=False)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layer_scale1): DINOv3ViTLayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): DINOv3ViTMLP(
          (up_proj): Linear(in_features=768, out_features=3072, bias=True)
          (down_proj): Linear(in_features=3072, out_fe

In [ ]:
@torch.inference_mode()
def embed_images_dinov3(images, batch_size=16):
    """
    images: list[PIL.Image]
    returns: torch.Tensor of shape [len(images), embedding_dim], L2-normalized
    """
    all_vecs = []

    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size]

        inputs = processor(
            images=batch,
            return_tensors="pt",
        ).to(device)

        outputs = dino_model(**inputs)

        vecs = outputs.pooler_output
        vecs = F.normalize(vecs, p=2, dim=-1)

        all_vecs.append(vecs.cpu())

    return torch.cat(all_vecs, dim=0)

In [ ]:
def compute_shot_embeddings(shots, video_path):
  reader = FrameReader(video_path)

  shot_vectors = []
  shot_frame_indices = []

  try:
      for _, row in tqdm(shots.iterrows(), total=len(shots)):
          frame_indices = [
              int(row["frame_25"]),
              int(row["frame_50"]),
              int(row["frame_75"]),
          ]

          frames = [reader.read(idx) for idx in frame_indices]

          frame_vecs = embed_images_dinov3(frames, batch_size=3)

          shot_vec = frame_vecs.mean(dim=0)
          shot_vec = F.normalize(shot_vec, p=2, dim=0)

          shot_vectors.append(shot_vec.numpy())
          shot_frame_indices.append(frame_indices)

  finally:
      reader.close()

  return shot_vectors, shot_frame_indices

# Agglomerative Clustering Setup

First, we do the adjacent-only / temporally constrained grouping using a connectivity matrix. The connectivity constraints allow only neighboring samples to be merged and use a distance_threshold so clustering can stop when linkage distance exceeds a threshold.

In [ ]:
def make_chain_connectivity(n: int) -> csr_matrix:
    """
    Connect each shot only to the immediately preceding and following shot.
    """
    if n == 1:
        return csr_matrix((1, 1), dtype=float)

    return diags(
        diagonals=[
            np.ones(n - 1),
            np.ones(n - 1),
        ],
        offsets=[-1, 1],
        shape=(n, n),
        format="csr",
        dtype=float,
    )


def make_visual_temporal_distance_matrix(
    shot_vectors: np.ndarray,
    shot_mid_frames: np.ndarray,
    video_num_frames: int,
    w_visual: float = 0.9,
    w_time: float = 0.1,
) -> np.ndarray:
    """
    D[i, j] =
        w_visual * cosine_distance(i, j)
        + w_time * normalized_temporal_distance(i, j)
    """
    vectors = np.asarray(shot_vectors, dtype=np.float64)
    mid_frames = np.asarray(shot_mid_frames, dtype=np.float64)

    norms = np.linalg.norm(vectors, axis=1)
    visual_distances = cosine_distances(vectors)
    temporal_distances = (
        np.abs(mid_frames[:, None] - mid_frames[None, :])
        / float(video_num_frames)
    )

    distances = (
        w_visual * visual_distances
        + w_time * temporal_distances
    )

    distances = 0.5 * (distances + distances.T)
    np.fill_diagonal(distances, 0.0)

    return distances


In [ ]:
def labels_to_scenes(df, labels):
    rows = []

    scene_id = 0
    start_shot = 0
    current_label = labels[0]

    for i in range(1, len(labels)):
        if labels[i] != current_label:
            end_shot = i - 1

            rows.append({
                "scene_id": scene_id,
                "start_shot": start_shot,
                "end_shot": end_shot,
                "num_shots": end_shot - start_shot + 1,
                "start_frame": int(df.loc[start_shot, "start_frame"]),
                "end_frame": int(df.loc[end_shot, "end_frame"]),
            })

            scene_id += 1
            start_shot = i
            current_label = labels[i]

    end_shot = len(labels) - 1
    rows.append({
        "scene_id": scene_id,
        "start_shot": start_shot,
        "end_shot": end_shot,
        "num_shots": end_shot - start_shot + 1,
        "start_frame": int(df.loc[start_shot, "start_frame"]),
        "end_frame": int(df.loc[end_shot, "end_frame"]),
    })

    return pd.DataFrame(rows)

### Adjacent-only Complete Linkage Setup

In [ ]:
from heapq import heappop, heappush


def cosine_distance_matrix(shot_vecs: np.ndarray) -> np.ndarray:
    """
    Construct a cosine-distance matrix from shot embeddings.

    Distance:
        D[i, j] = 1 - cosine_similarity(i, j)

    Raises:
        ValueError:
            If embeddings are not a finite 2-D array or contain zero vectors.
    """
    vectors = np.asarray(shot_vecs, dtype=np.float64)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)

    if np.any(norms == 0):
        zero_indices = np.flatnonzero(norms[:, 0] == 0)
        raise ValueError(
            f"Shot embeddings must be nonzero. "
            f"Zero vectors found at indices {zero_indices.tolist()}."
        )

    normalized = vectors / norms
    similarities = normalized @ normalized.T
    similarities = np.clip(similarities, -1.0, 1.0)

    distances = 1.0 - similarities
    np.fill_diagonal(distances, 0.0)

    return distances


def temporal_complete_linkage(
    distance_matrix: np.ndarray,
    distance_threshold: float,
    boundary_prior: np.ndarray | None = None,
    boundary_weight: float = 1.0,
) -> tuple[np.ndarray, list[tuple[int, int]]]:
    """
    Temporally constrained complete-linkage agglomerative clustering.

    Every shot begins as its own cluster. Only temporally adjacent clusters
    may merge.

    The merge score for adjacent clusters A and B is:

        score(A, B)
            = max(D[i, j] for i in A for j in B)
            + boundary_weight * boundary_prior[b]

    where b is the original boundary after the final shot in A.

    A merge occurs only when:

        score(A, B) < distance_threshold

    Therefore, a score exactly equal to the threshold is not merged.

    Args:
        distance_matrix:
            Symmetric shot-to-shot distance matrix with shape (n, n).

            Positive infinity is allowed and acts as a hard constraint. For
            example, setting selected entries to infinity prevents any cluster
            containing those shot pairs from merging under complete linkage.

        distance_threshold:
            Maximum combined merge score. Must be finite and nonnegative.

        boundary_prior:
            Optional array with shape (n - 1,). Entry k represents evidence
            that the boundary between shots k and k + 1 is a scene boundary.

            Larger values discourage merging across that boundary. Positive
            infinity creates a hard scene boundary.

            When omitted, all boundary penalties are zero.

        boundary_weight:
            Nonnegative multiplier controlling the influence of the boundary
            prior.

    Returns:
        labels:
            Integer scene label for every shot, ordered temporally.

        scene_ranges:
            Inclusive (start_shot_index, end_shot_index) pairs.

    Notes:
        Because only adjacent clusters merge, every cluster remains a
        contiguous interval. The implementation therefore stores only each
        cluster's start and end indices rather than an explicit member list.
    """
    distances = np.asarray(distance_matrix, dtype=np.float64)


    if not np.allclose(
        distances,
        distances.T,
        rtol=1e-10,
        atol=1e-10,
    ):
        raise ValueError("distance_matrix must be symmetric.")

    n = distances.shape[0]

    if boundary_prior is None:
        boundary_prior_array = np.zeros(max(0, n - 1), dtype=np.float64)
    else:
        boundary_prior_array = np.asarray(
            boundary_prior,
            dtype=np.float64,
        )

        expected_shape = (max(0, n - 1),)

    if n == 0:
        return np.empty(0, dtype=int), []

    # Each active cluster is a contiguous interval [start[id], end[id]].
    start = {i: i for i in range(n)}
    end = {i: i for i in range(n)}

    # Doubly linked list maintaining temporal cluster order.
    left = {
        i: i - 1 if i > 0 else None
        for i in range(n)
    }
    right = {
        i: i + 1 if i < n - 1 else None
        for i in range(n)
    }

    active = set(range(n))

    # Heap entries:
    #
    #   (
    #       combined_merge_score,
    #       temporal_start_for_tie_breaking,
    #       left_cluster_id,
    #       right_cluster_id,
    #   )
    #
    # The temporal start makes equal-score behavior deterministic and favors
    # the leftmost candidate.
    merge_heap: list[tuple[float, int, int, int]] = []

    def complete_linkage_cost(
        left_id: int,
        right_id: int,
    ) -> float:
        """
        True complete-linkage distance between two contiguous clusters.
        """
        cross_distances = distances[
            start[left_id] : end[left_id] + 1,
            start[right_id] : end[right_id] + 1,
        ]

        return float(np.max(cross_distances))

    def boundary_penalty(left_id: int) -> float:
        """
        Penalty for the boundary after the final shot of the left cluster.
        """
        if boundary_weight == 0:
            # Avoid 0 * infinity producing NaN.
            return 0.0

        boundary_index = end[left_id]

        return float(
            boundary_weight * boundary_prior_array[boundary_index]
        )

    def merge_score(
        left_id: int,
        right_id: int,
    ) -> float:
        return (
            complete_linkage_cost(left_id, right_id)
            + boundary_penalty(left_id)
        )

    def push_candidate(
        left_id: int,
        right_id: int,
    ) -> None:
        score = merge_score(left_id, right_id)

        heappush(
            merge_heap,
            (
                score,
                start[left_id],
                left_id,
                right_id,
            ),
        )

    # Initially, only consecutive shots are valid merge candidates.
    for shot_index in range(n - 1):
        push_candidate(shot_index, shot_index + 1)

    next_cluster_id = n

    while merge_heap:
        score, _, left_id, right_id = heappop(merge_heap)

        # Heap entries become stale when either participating cluster has
        # already been merged or when the two clusters are no longer adjacent.
        if (
            left_id not in active
            or right_id not in active
            or right.get(left_id) != right_id
            or left.get(right_id) != left_id
        ):
            continue

        # Strict threshold semantics:
        #
        #     merge only when score < distance_threshold
        #
        # Since this is the smallest valid heap entry, no remaining candidate
        # can merge once this condition is reached.
        if score >= distance_threshold:
            break

        new_cluster_id = next_cluster_id
        next_cluster_id += 1

        outside_left = left[left_id]
        outside_right = right[right_id]

        # The two clusters are adjacent, so their union remains contiguous.
        start[new_cluster_id] = start[left_id]
        end[new_cluster_id] = end[right_id]

        left[new_cluster_id] = outside_left
        right[new_cluster_id] = outside_right

        active.remove(left_id)
        active.remove(right_id)
        active.add(new_cluster_id)

        if outside_left is not None:
            right[outside_left] = new_cluster_id
            push_candidate(outside_left, new_cluster_id)

        if outside_right is not None:
            left[outside_right] = new_cluster_id
            push_candidate(new_cluster_id, outside_right)

    # Locate the first remaining cluster.
    first_cluster = next(
        cluster_id
        for cluster_id in active
        if left[cluster_id] is None
    )

    labels = np.empty(n, dtype=int)
    scene_ranges: list[tuple[int, int]] = []

    scene_id = 0
    cluster_id: int | None = first_cluster

    while cluster_id is not None:
        scene_start = start[cluster_id]
        scene_end = end[cluster_id]

        labels[scene_start : scene_end + 1] = scene_id
        scene_ranges.append((scene_start, scene_end))

        scene_id += 1
        cluster_id = right[cluster_id]

    return labels, scene_ranges

#### Shot Transition Boundary Prior Setup

As part of establishing a baseline we use static penalties for shot transition labels. OmniShotCut has two modes clean_shot and default mode. The clean_shot mode filters out non-General (e.g. Dissolve) ranges, while default mode returns those ranges and labels. However, those regions should not be independent semantic shot nodes during clustering.  We run the default in order to retain the transition metadata, but then remove transition-only nodes and attach that metadata to the boundary between the neighboring clean shots.



In [ ]:
# OmniShotCut integer-to-label mappings.
OMNISHOTCUT_INTRA_LABELS = {
    0: "General",
    1: "Dissolve",
    2: "Wipes",
    3: "Push",
    4: "Slide",
    5: "Zoom",
    6: "Fade",
    7: "Doorway",
    8: "Padding",
}

OMNISHOTCUT_INTER_LABELS = {
    0: "New_Start",
    1: "Hard_Cut",
    2: "Transition_Source",
    3: "Transition",
    4: "Sudden_Jump",
    5: "Padding",
}

# These represent relative evidence that a semantic scene boundary exists. These
# were somewhat arbitrarily set.
DEFAULT_INTRA_BOUNDARY_PENALTIES = {
    "General": 0.00,
    "Dissolve": 0.55,
    "Wipes": 0.30,
    "Push": 0.20,
    "Slide": 0.20,
    "Zoom": 0.10,
    "Fade": 0.75,
    "Doorway": 0.30,
}

DEFAULT_INTER_BOUNDARY_PENALTIES = {
    "New_Start": 0.00,
    "Hard_Cut": 0.05,
    "Transition_Source": 0.10,
    "Transition": 0.55,
    "Sudden_Jump": 0.05,
}


def _normalize_omnishotcut_label(
    label,
    integer_mapping: Mapping[int, str],
    valid_names: set[str],
) -> str:
    """
    Convert an OmniShotCut integer or string label to its canonical name.
    """
    if isinstance(label, (int, np.integer)):
        label_id = int(label)

        if label_id not in integer_mapping:
            raise ValueError(f"Unknown OmniShotCut label ID: {label_id}.")

        return integer_mapping[label_id]

    raw_name = str(label).strip()

    # Support canonical names and common capitalization variations.
    normalized_lookup = {
        name.lower(): name
        for name in valid_names
    }

    key = raw_name.lower()

    if key not in normalized_lookup:
        raise ValueError(f"Unknown OmniShotCut label: {raw_name!r}.")

    return normalized_lookup[key]


def make_static_omnishotcut_boundary_prior(
    ranges: Sequence[Sequence[int]],
    intra_labels: Sequence[int | str],
    inter_labels: Sequence[int | str],
    *,
    intra_penalties: Mapping[str, float] | None = None,
    inter_penalties: Mapping[str, float] | None = None,
    combine: str = "max",
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    list[dict[str, Any]],
]:
    """
    Convert OmniShotCut default-mode output into clean shot ranges and one
    static prior per boundary between consecutive clean shots.

    Non-General ranges such as Fade or Dissolve are treated as transition
    metadata rather than independent semantic shots.

    For two consecutive clean shots, all OmniShotCut labels occurring between
    them are mapped to penalties. These penalties are combined into one prior.

    Args:
        ranges:
            OmniShotCut ranges returned by mode="default", shape (n, 2).

        intra_labels:
            One intra-shot label per range.

        inter_labels:
            One inter-shot label per range. The label at index i describes
            range i relative to the preceding range.

        intra_penalties:
            Optional override for the static intra-label mapping.

        inter_penalties:
            Optional override for the static inter-label mapping.

        combine:
            How to combine evidence between two clean shots:

            "max":
                Use the strongest observed label. Recommended initially,
                because intra and inter predictions are correlated.

            "mean":
                Average all observed nonzero penalties.

            "sum_clipped":
                Add penalties and clip the result to [0, 1].

    Returns:
        clean_ranges:
            Only ranges whose intra-label is General, shape (num_clean, 2).

        clean_raw_indices:
            Indices of those clean ranges in the original OmniShotCut output.

        boundary_prior:
            Array of shape (num_clean - 1,). Entry b corresponds to the
            boundary between clean shots b and b + 1.

        boundary_metadata:
            Diagnostic information describing how each prior was constructed.
    """
    ranges_array = np.asarray(ranges, dtype=np.int64)

    if ranges_array.ndim != 2 or ranges_array.shape[1] != 2:
        raise ValueError("ranges must have shape (num_ranges, 2).")

    num_ranges = len(ranges_array)

    if len(intra_labels) != num_ranges:
        raise ValueError(
            "intra_labels must contain one label per range."
        )

    if len(inter_labels) != num_ranges:
        raise ValueError(
            "inter_labels must contain one label per range."
        )

    if combine not in {"max", "mean", "sum_clipped"}:
        raise ValueError(
            "combine must be 'max', 'mean', or 'sum_clipped'."
        )

    intra_penalties = dict(
        DEFAULT_INTRA_BOUNDARY_PENALTIES
        if intra_penalties is None
        else intra_penalties
    )

    inter_penalties = dict(
        DEFAULT_INTER_BOUNDARY_PENALTIES
        if inter_penalties is None
        else inter_penalties
    )

    for mapping_name, mapping in (
        ("intra_penalties", intra_penalties),
        ("inter_penalties", inter_penalties),
    ):
        for label, penalty in mapping.items():
            penalty = float(penalty)

            if not np.isfinite(penalty) or penalty < 0:
                raise ValueError(
                    f"{mapping_name}[{label!r}] must be finite "
                    "and nonnegative."
                )

    valid_intra_names = set(OMNISHOTCUT_INTRA_LABELS.values())
    valid_inter_names = set(OMNISHOTCUT_INTER_LABELS.values())

    intra_names = [
        _normalize_omnishotcut_label(
            label,
            OMNISHOTCUT_INTRA_LABELS,
            valid_intra_names,
        )
        for label in intra_labels
    ]

    inter_names = [
        _normalize_omnishotcut_label(
            label,
            OMNISHOTCUT_INTER_LABELS,
            valid_inter_names,
        )
        for label in inter_labels
    ]

    # OmniShotCut default mode can emit batch-padding ranges labeled "Padding".
    # These are artifacts, not real shots or transitions, so drop them before
    # computing clean shots and boundary priors.
    keep_indices = [
        i
        for i in range(num_ranges)
        if intra_names[i] != "Padding" and inter_names[i] != "Padding"
    ]
    ranges_array = ranges_array[keep_indices]
    intra_names = [intra_names[i] for i in keep_indices]
    inter_names = [inter_names[i] for i in keep_indices]

    clean_raw_indices = np.asarray(
        [
            index
            for index, label in enumerate(intra_names)
            if label == "General"
        ],
        dtype=int,
    )

    if clean_raw_indices.size == 0:
        raise ValueError(
            "No General ranges were found in the OmniShotCut output."
        )

    clean_ranges = ranges_array[clean_raw_indices]

    if clean_raw_indices.size == 1:
        return (
            clean_ranges,
            clean_raw_indices,
            np.empty(0, dtype=np.float64),
            [],
        )

    boundary_priors: list[float] = []
    metadata: list[dict[str, Any]] = []

    for clean_boundary_index in range(clean_raw_indices.size - 1):
        left_raw_index = int(
            clean_raw_indices[clean_boundary_index]
        )
        right_raw_index = int(
            clean_raw_indices[clean_boundary_index + 1]
        )

        # Each range after the left clean range, through the right clean
        # range, has an inter-label describing its relationship to its
        # predecessor.
        intervening_inter_labels = inter_names[
            left_raw_index + 1 : right_raw_index + 1
        ]

        # Non-General ranges strictly between the two clean shots describe
        # transition effects such as a fade or dissolve.
        intervening_intra_labels = intra_names[
            left_raw_index + 1 : right_raw_index
        ]

        evidence: list[tuple[str, str, float]] = []

        for label in intervening_inter_labels:
            penalty = float(inter_penalties.get(label, 0.0))

            evidence.append(
                ("inter", label, penalty)
            )

        for label in intervening_intra_labels:
            penalty = float(intra_penalties.get(label, 0.0))

            evidence.append(
                ("intra", label, penalty)
            )

        evidence_values = [
            penalty
            for _, _, penalty in evidence
            if penalty > 0
        ]

        if not evidence_values:
            prior = 0.0
        elif combine == "max":
            prior = max(evidence_values)
        elif combine == "mean":
            prior = float(np.mean(evidence_values))
        else:
            prior = min(1.0, float(np.sum(evidence_values)))

        boundary_priors.append(prior)

        strongest_evidence = (
            max(evidence, key=lambda item: item[2])
            if evidence
            else None
        )

        metadata.append(
            {
                "clean_boundary_index": clean_boundary_index,
                "left_clean_shot": clean_boundary_index,
                "right_clean_shot": clean_boundary_index + 1,
                "left_raw_index": left_raw_index,
                "right_raw_index": right_raw_index,
                "left_range": clean_ranges[
                    clean_boundary_index
                ].tolist(),
                "right_range": clean_ranges[
                    clean_boundary_index + 1
                ].tolist(),
                "inter_labels": intervening_inter_labels,
                "intra_labels": intervening_intra_labels,
                "evidence": evidence,
                "strongest_evidence": strongest_evidence,
                "boundary_prior": prior,
            }
        )

    return (
        clean_ranges,
        clean_raw_indices,
        np.asarray(boundary_priors, dtype=np.float64),
        metadata,
    )

### Establishing-Shot Prior

Run ShotVL-7B over each movie's **clean** shots (video clip + shot-size + location) and
cache `is_establishing = is_wide & is_location` per clean shot to
`{movie}_establishing.npy`. Expensive on first run (2 generations per shot across all
movies), cached thereafter. Consumed by the `establishing` pipeline in Evaluate.
The **guarded loader** below defines the ShotVL helpers and loads the 7B model only on a cache miss, so a normal run (labels already cached) never loads it.

In [ ]:
# Guarded ShotVL-7B loader.
#
# Defines the establishing-shot helpers cheaply. The 7B model is loaded ONLY on
# first actual use (a cache miss in compute_establishing_labels below); if every
# {movie}_establishing.npy already exists, the model is never loaded.
#
# To REGENERATE labels in a fresh session, first install the deps:
# !pip install -q -U "qwen-vl-utils" "transformers>=4.49.0" accelerate
import os
import tempfile

SHOTVL_ID = "Vchitect/ShotVL-7B"
_SHOTVL = {"model": None, "processor": None}

SHOT_SIZE_OPTIONS = [
    "Extreme wide shot",
    "Wide shot",
    "Medium shot",
    "Close-up",
    "Clean single",
    "Profile shot",
]
WIDE_SIZES = {"extreme wide shot", "wide shot"}

SIZE_Q = (
    "What is the shot size of this shot?\n"
    "Options:\n"
    "A. Extreme wide shot\n"
    "B. Wide shot\n"
    "C. Medium shot\n"
    "D. Close-up\n"
    "E. Clean single\n"
    "F. Profile shot\n"
    "Answer with the option letter and label."
)
LOCATION_Q = (
    "Does this shot primarily depict a location, setting, or environment, "
    "rather than focusing on a person or people? Answer yes or no."
)


def _load_shotvl():
    """Load ShotVL-7B once, lazily, on first use."""
    if _SHOTVL["model"] is None:
        import torch
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

        print("Loading ShotVL-7B (first use)...")
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            SHOTVL_ID,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            attn_implementation="sdpa",
        )
        model.eval()
        _SHOTVL["model"] = model
        _SHOTVL["processor"] = AutoProcessor.from_pretrained(SHOTVL_ID)
    return _SHOTVL["model"], _SHOTVL["processor"]


def sample_shot_frame_paths(reader, start_frame, end_frame, tmpdir, num_frames=8):
    """Uniformly sample num_frames from a shot and write them as PNGs for video mode."""
    start_frame = int(start_frame)
    end_frame = int(end_frame)
    last = max(start_frame, end_frame - 1)
    frame_indices = np.linspace(start_frame, last, num=num_frames).round().astype(int)
    paths = []
    for k, idx in enumerate(frame_indices):
        path = os.path.join(tmpdir, f"f_{k:03d}.png")
        reader.read(int(idx)).save(path)
        paths.append("file://" + path)
    return paths


def ask_shotvl_video(frame_paths, question, max_new_tokens=64):
    """Ask ShotVL a question about a shot presented as sampled video frames."""
    import torch
    from qwen_vl_utils import process_vision_info

    model, processor = _load_shotvl()
    msgs = [
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "video", "video": frame_paths, "fps": 4.0},
                {"type": "text", "text": question},
            ],
        },
    ]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(msgs)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()


def parse_shot_size(text):
    lowered = text.lower()
    for label in SHOT_SIZE_OPTIONS:
        if label.lower() in lowered:
            return label
    for letter, label in zip("abcdef", SHOT_SIZE_OPTIONS):
        if lowered.startswith(letter):
            return label
    return "Unknown"


def parse_yes(text):
    return text.strip().lower().startswith("y")

In [ ]:
def _clean_ranges_for_movie(movie):
    shots = omni_shots_all[omni_shots_all["movie_name"] == movie].reset_index(drop=True)
    clean_ranges, _, _, _ = make_static_omnishotcut_boundary_prior(
        ranges=shots[["start_frame", "end_frame"]].to_numpy().tolist(),
        intra_labels=shots["intra_label"].tolist(),
        inter_labels=shots["inter_label"].tolist(),
        combine="max",
    )
    return clean_ranges


def compute_establishing_labels(movie):
    """is_wide & is_location per clean shot (ShotVL video mode), cached to Drive."""
    cache = OMNI_DIR / f"{movie}_establishing.npy"
    if cache.exists():
        return np.load(cache)

    clean_ranges = _clean_ranges_for_movie(movie)
    reader = FrameReader(str(BBC_ROOT / f"{movie}.mp4"))
    flags = []
    try:
        with tempfile.TemporaryDirectory() as base_tmp:
            for k, (start_frame, end_frame) in enumerate(
                tqdm(clean_ranges, desc=f"establishing:{movie}", leave=False)
            ):
                shot_tmp = os.path.join(base_tmp, f"s_{k:05d}")
                os.makedirs(shot_tmp, exist_ok=True)
                frames = sample_shot_frame_paths(reader, start_frame, end_frame, shot_tmp)
                size = parse_shot_size(ask_shotvl_video(frames, SIZE_Q))
                location = parse_yes(ask_shotvl_video(frames, LOCATION_Q))
                flags.append((size.lower() in WIDE_SIZES) and location)
    finally:
        reader.close()

    flags = np.asarray(flags, dtype=bool)
    np.save(cache, flags)
    return flags


for _movie in tqdm(bbc_movies, desc="establishing labels"):
    compute_establishing_labels(_movie)

establishing labels:   0%|          | 0/11 [00:00<?, ?it/s]

# Evaluate

Scene detection is scored as **frame-tolerant boundary detection** over the whole BBC
dataset. A predicted scene boundary (a scene's `start_frame`) is correct when it falls
within `+/- TOL_FRAMES` of a ground-truth boundary; matching is 1-to-1.

* **F1** — precision / recall / F1 of the predicted (hard) complete-linkage scene
  boundaries, matched to GT within tolerance.
* **AP** (Average Precision) — every adjacent clean-shot boundary is a candidate,
  scored by its complete-linkage boundary cost (adjacent cosine distance +
  `boundary_weight * boundary_prior`). Candidates are ranked by score and matched to GT
  within tolerance to trace a precision-recall curve; AP is its area.

Reported per movie and aggregated as macro means (mAP / mean-F1) across all 11 movies.

Both grouping methods are scored side by side via a `method` column: **complete_linkage** (clean/General shots, cosine + boundary prior) and **agglomerative** (all shots, visual + temporal distance).

A third method — **establishing** — is complete_linkage with a strong boundary-prior boost before each ShotVL-detected establishing shot.

In [ ]:
def boundary_f1(pred_scene_df, gt_scene_df, tol_frames, frame_col="start_frame"):
    """Frame-tolerant scene-boundary precision/recall/F1 (1-to-1 nearest matching)."""
    pred_b = np.sort(pred_scene_df[frame_col].to_numpy().astype(int))[1:]
    gt_b = np.sort(gt_scene_df[frame_col].to_numpy().astype(int))[1:]

    pairs = sorted(
        (
            (abs(int(p) - int(g)), i, j)
            for i, p in enumerate(pred_b)
            for j, g in enumerate(gt_b)
            if abs(int(p) - int(g)) <= tol_frames
        ),
        key=lambda t: t[0],
    )

    used_p, used_g = set(), set()
    tp = 0
    for _, i, j in pairs:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        tp += 1

    fp = len(pred_b) - tp
    fn = len(gt_b) - tp
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "num_pred_boundaries": int(len(pred_b)),
        "num_gt_boundaries": int(len(gt_b)),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def boundary_average_precision(pred_frames, pred_scores, gt_frames, tol_frames):
    """Frame-tolerant boundary Average Precision.

    Candidate boundaries are ranked by score (desc) and greedily matched 1-to-1 to the
    nearest unmatched GT boundary within +/- tol_frames. AP is the area under the
    resulting precision-recall curve (sum of precision at each true positive / num_gt).
    """
    pred_frames = np.asarray(pred_frames, dtype=float)
    pred_scores = np.asarray(pred_scores, dtype=float)
    gt = np.sort(np.asarray(gt_frames, dtype=float))
    num_gt = len(gt)

    if num_gt == 0:
        return float("nan")
    if len(pred_frames) == 0:
        return 0.0

    order = np.argsort(-pred_scores)
    ranked_frames = pred_frames[order]

    matched = np.zeros(num_gt, dtype=bool)
    tp = np.zeros(len(ranked_frames))
    fp = np.zeros(len(ranked_frames))

    for k, frame in enumerate(ranked_frames):
        best_j, best_d = -1, tol_frames + 1
        for j in range(num_gt):
            if matched[j]:
                continue
            d = abs(frame - gt[j])
            if d <= tol_frames and d < best_d:
                best_d, best_j = d, j
        if best_j >= 0:
            matched[best_j] = True
            tp[k] = 1.0
        else:
            fp[k] = 1.0

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)

    return float(np.sum(precision[tp.astype(bool)]) / num_gt)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

TOL_FRAMES = 2
PADDING_INTRA_CODE = 8  # OmniShotCut intra "Padding"
PADDING_INTER_CODE = 5  # OmniShotCut inter "Padding"
ESTABLISHING_BOOST = 1.0  # soft prior added before each establishing shot


def _add_frame_samples(shots):
    shots = shots.reset_index(drop=True).copy()
    sampled = shots.apply(
        lambda r: sample_interior_frame_indices(
            r["start_frame"], r["end_frame"], end_inclusive=False
        ),
        axis=1,
    )
    shots[["frame_25", "frame_50", "frame_75"]] = pd.DataFrame(
        sampled.tolist(), index=shots.index
    )
    return shots


def _movie_shots(movie, drop_padding=True):
    shots = omni_shots_all[omni_shots_all["movie_name"] == movie].reset_index(drop=True)
    if drop_padding:
        is_padding = (
            shots["intra_label"].isin([PADDING_INTRA_CODE, "Padding"])
            | shots["inter_label"].isin([PADDING_INTER_CODE, "Padding"])
        )
        shots = shots[~is_padding].reset_index(drop=True)
    return shots


def _cached_embeddings(shots, movie, cache_name):
    cache = OMNI_DIR / f"{movie}_{cache_name}.npy"
    if cache.exists():
        return np.load(cache)
    vecs, _ = compute_shot_embeddings(shots, str(BBC_ROOT / f"{movie}.mp4"))
    vectors = np.asarray(vecs)
    np.save(cache, vectors)
    return vectors


def _boundary_candidates(distance_matrix, frames, extra=None):
    n = len(frames)
    if n < 2:
        return np.empty(0), np.empty(0)
    adjacent = np.array([distance_matrix[i, i + 1] for i in range(n - 1)])
    scores = adjacent if extra is None else adjacent + extra
    return frames[1:], scores


def process_movie_complete_linkage(movie, distance_threshold=0.8, boundary_weight=0.25):
    """Adjacent-only complete linkage on CLEAN (General) shots, cosine + boundary prior."""
    shots = _movie_shots(movie, drop_padding=False)

    clean_ranges, _, boundary_prior, _ = make_static_omnishotcut_boundary_prior(
        ranges=shots[["start_frame", "end_frame"]].to_numpy().tolist(),
        intra_labels=shots["intra_label"].tolist(),
        inter_labels=shots["inter_label"].tolist(),
        combine="max",
    )
    clean_shots = _add_frame_samples(
        pd.DataFrame(
            {
                "shot_index": np.arange(len(clean_ranges)),
                "start_frame": clean_ranges[:, 0],
                "end_frame": clean_ranges[:, 1],
            }
        )
    )

    clean_vectors = _cached_embeddings(clean_shots, movie, "clean_vectors")
    distance_matrix = cosine_distance_matrix(clean_vectors)

    labels, _ = temporal_complete_linkage(
        distance_matrix=distance_matrix,
        distance_threshold=distance_threshold,
        boundary_prior=boundary_prior,
        boundary_weight=boundary_weight,
    )
    scene_df = labels_to_scenes(clean_shots, labels)

    frames = clean_shots["start_frame"].to_numpy()
    extra = boundary_weight * np.asarray(boundary_prior) if len(frames) >= 2 else None
    cand_frames, cand_scores = _boundary_candidates(distance_matrix, frames, extra)
    return {"scene_df": scene_df, "cand_frames": cand_frames, "cand_scores": cand_scores}


def process_movie_agglomerative(movie, distance_threshold=0.5, w_visual=0.7, w_time=0.3):
    """sklearn adjacent-only agglomerative on ALL shots, visual + temporal distance."""
    shots = _add_frame_samples(_movie_shots(movie, drop_padding=True))
    vectors = _cached_embeddings(shots, movie, "all_vectors")

    n = len(shots)
    mid_frames = (
        shots["start_frame"].to_numpy() + shots["end_frame"].to_numpy()
    ) / 2.0
    video_num_frames = int(shots["end_frame"].max() + 1)

    distance_matrix = make_visual_temporal_distance_matrix(
        shot_vectors=vectors,
        shot_mid_frames=mid_frames,
        video_num_frames=video_num_frames,
        w_visual=w_visual,
        w_time=w_time,
    )
    connectivity = make_chain_connectivity(n)

    agg_model = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric="precomputed",
        linkage="complete",
        connectivity=connectivity,
        compute_full_tree=True,
        compute_distances=True,
    )
    labels = agg_model.fit_predict(distance_matrix)
    scene_df = labels_to_scenes(shots, labels)

    cand_frames, cand_scores = _boundary_candidates(
        distance_matrix, shots["start_frame"].to_numpy()
    )
    return {"scene_df": scene_df, "cand_frames": cand_frames, "cand_scores": cand_scores}


def process_movie_establishing(
    movie, distance_threshold=0.8, boundary_weight=0.25, establishing_boost=ESTABLISHING_BOOST
):
    """Complete linkage + a strong boundary-prior boost before each establishing shot."""
    shots = _movie_shots(movie, drop_padding=False)

    clean_ranges, _, boundary_prior, _ = make_static_omnishotcut_boundary_prior(
        ranges=shots[["start_frame", "end_frame"]].to_numpy().tolist(),
        intra_labels=shots["intra_label"].tolist(),
        inter_labels=shots["inter_label"].tolist(),
        combine="max",
    )
    clean_shots = _add_frame_samples(
        pd.DataFrame(
            {
                "shot_index": np.arange(len(clean_ranges)),
                "start_frame": clean_ranges[:, 0],
                "end_frame": clean_ranges[:, 1],
            }
        )
    )

    clean_vectors = _cached_embeddings(clean_shots, movie, "clean_vectors")
    distance_matrix = cosine_distance_matrix(clean_vectors)

    # Boost the boundary BEFORE each establishing shot so it is favored to start a scene.
    establishing = compute_establishing_labels(movie).astype(bool)
    if len(establishing) != len(clean_shots):
        raise ValueError(
            f"establishing labels ({len(establishing)}) != clean shots "
            f"({len(clean_shots)}) for {movie}; delete {movie}_establishing.npy and rerun."
        )
    prior = np.asarray(boundary_prior, dtype=float).copy()
    for shot_i in np.flatnonzero(establishing):
        if shot_i >= 1:  # boundary shot_i-1 sits just before clean shot shot_i
            prior[shot_i - 1] += establishing_boost

    labels, _ = temporal_complete_linkage(
        distance_matrix=distance_matrix,
        distance_threshold=distance_threshold,
        boundary_prior=prior,
        boundary_weight=boundary_weight,
    )
    scene_df = labels_to_scenes(clean_shots, labels)

    frames = clean_shots["start_frame"].to_numpy()
    extra = boundary_weight * prior if len(frames) >= 2 else None
    cand_frames, cand_scores = _boundary_candidates(distance_matrix, frames, extra)
    return {"scene_df": scene_df, "cand_frames": cand_frames, "cand_scores": cand_scores}


def evaluate_movie(movie, result, tol_frames=TOL_FRAMES):
    gt = bbc_scenes_gt[bbc_scenes_gt["movie_name"] == movie].dropna(subset=["start_frame"])
    gt_frames = np.sort(gt["start_frame"].to_numpy().astype(int))[1:]

    bf1 = boundary_f1(result["scene_df"], gt, tol_frames)
    ap = boundary_average_precision(
        result["cand_frames"], result["cand_scores"], gt_frames, tol_frames
    )
    return {
        "movie_name": movie,
        "num_pred_scenes": len(result["scene_df"]),
        "num_gt_scenes": len(gt),
        "num_gt_boundaries": bf1["num_gt_boundaries"],
        "tp": bf1["tp"],
        "fp": bf1["fp"],
        "fn": bf1["fn"],
        "precision": bf1["precision"],
        "recall": bf1["recall"],
        "f1": bf1["f1"],
        "ap": ap,
    }


METHODS = {
    "complete_linkage": process_movie_complete_linkage,
    "establishing": process_movie_establishing,
    "agglomerative": process_movie_agglomerative,
}

scene_dfs, rows = {}, []
for method_name, process_fn in METHODS.items():
    for movie in tqdm(bbc_movies, desc=method_name):
        result = process_fn(movie)
        scene_dfs[(method_name, movie)] = result["scene_df"]
        row = evaluate_movie(movie, result, tol_frames=TOL_FRAMES)
        row["method"] = method_name
        rows.append(row)

results_df = pd.DataFrame(rows)

mean_rows = []
for method_name in METHODS:
    means = (
        results_df[results_df["method"] == method_name]
        .select_dtypes("number")
        .mean()
    )
    means["method"] = method_name
    means["movie_name"] = "MEAN (macro)"
    mean_rows.append(means)
results_df = pd.concat([results_df, pd.DataFrame(mean_rows)], ignore_index=True)

lead = ["method", "movie_name"]
results_df = results_df[lead + [c for c in results_df.columns if c not in lead]]

display(results_df)

# Head-to-head summary: one row per method.
summary_df = (
    results_df[results_df["movie_name"] == "MEAN (macro)"][
        ["method", "ap", "f1", "precision", "recall"]
    ]
    .rename(
        columns={
            "ap": "mAP",
            "f1": "mean_f1",
            "precision": "mean_precision",
            "recall": "mean_recall",
        }
    )
    .reset_index(drop=True)
)
display(summary_df)

print(f"Frame-tolerant boundary metrics @ tol=+/-{TOL_FRAMES} frames over {len(bbc_movies)} movies")
for method_name in METHODS:
    row = results_df[
        (results_df["method"] == method_name)
        & (results_df["movie_name"] == "MEAN (macro)")
    ]
    print(f"  {method_name:18s} mAP={row['ap'].item():.4f}  meanF1={row['f1'].item():.4f}")

complete_linkage:   0%|          | 0/11 [00:00<?, ?it/s]

establishing:   0%|          | 0/11 [00:00<?, ?it/s]

agglomerative:   0%|          | 0/11 [00:00<?, ?it/s]

,method,movie_name,num_pred_scenes,num_gt_scenes,num_gt_boundaries,tp,fp,fn,precision,recall,f1,ap
0,complete_linkage,bbc_01,156.000000,66.000000,65.000000,23.000000,132.000000,42.000000,0.148387,0.353846,0.209091,0.142424
1,complete_linkage,bbc_02,115.000000,53.000000,52.000000,23.000000,91.000000,29.000000,0.201754,0.442308,0.277108,0.199088
2,complete_linkage,bbc_03,118.000000,62.000000,61.000000,37.000000,80.000000,24.000000,0.316239,0.606557,0.415730,0.343384
3,complete_linkage,bbc_04,152.000000,71.000000,70.000000,34.000000,117.000000,36.000000,0.225166,0.485714,0.307692,0.252276
4,complete_linkage,bbc_05,116.000000,65.000000,64.000000,38.000000,77.000000,26.000000,0.330435,0.593750,0.424581,0.283459
5,complete_linkage,bbc_06,177.000000,65.000000,64.000000,38.000000,138.000000,26.000000,0.215909,0.593750,0.316667,0.208592
6,complete_linkage,bbc_07,136.000000,62.000000,61.000000,44.000000,91.000000,17.000000,0.325926,0.721311,0.448980,0.361780
7,complete_linkage,bbc_08,103.000000,53.000000,52.000000,24.000000,78.000000,28.000000,0.235294,0.461538,0.311688,0.196807
8,complete_linkage,bbc_09,84.000000,61.000000,60.000000,43.000000,40.000000,17.000000,0.518072,0.716667,0.601399,0.513399
9,complete_linkage,bbc_10,98.000000,57.000000,56.000000,29.000000,68.000000,27.000000,0.298969,0.517857,0.379085,0.337682


,method,mAP,mean_f1,mean_precision,mean_recall
0,complete_linkage,0.280893,0.367384,0.278236,0.554946
1,establishing,0.301234,0.334994,0.225988,0.671307
2,agglomerative,0.289909,0.351523,0.276910,0.507613


Frame-tolerant boundary metrics @ tol=+/-2 frames over 11 movies
  complete_linkage   mAP=0.2809  meanF1=0.3674
  establishing       mAP=0.3012  meanF1=0.3350
  agglomerative      mAP=0.2899  meanF1=0.3515


## Tuning: threshold sweep & establishing repair

**(2) Over-segmentation.** Precision < recall means too many predicted boundaries.
Sweep `distance_threshold` (higher merges more -> fewer, higher-precision boundaries)
and pick the F1-max per method. Embeddings are cached, so this is fast.

**(3) Establishing prior.** `ESTABLISHING_BOOST` is now 1.0 (soft). We also sweep the
boost (boost=0 reproduces `complete_linkage`) and report the ShotVL detector's own
precision/recall as a scene-start predictor, to see whether the signal is worth keeping.

In [ ]:
def _sweep_threshold(process_fn, thresholds, tol_frames=TOL_FRAMES, **kwargs):
    rows = []
    for thr in tqdm(thresholds, desc=getattr(process_fn, "__name__", "sweep")):
        f1s, aps = [], []
        for movie in bbc_movies:
            ev = evaluate_movie(
                movie, process_fn(movie, distance_threshold=thr, **kwargs), tol_frames
            )
            f1s.append(ev["f1"])
            aps.append(ev["ap"])
        rows.append(
            {
                "distance_threshold": thr,
                "mean_f1": float(np.mean(f1s)),
                "mAP": float(np.mean(aps)),
            }
        )
    return pd.DataFrame(rows)


CL_GRID = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2]      # cosine-based methods
AGG_GRID = [0.3, 0.4, 0.5, 0.6, 0.7]          # visual+temporal distance

threshold_sweeps = {
    "complete_linkage": _sweep_threshold(process_movie_complete_linkage, CL_GRID),
    "establishing": _sweep_threshold(process_movie_establishing, CL_GRID),
    "agglomerative": _sweep_threshold(process_movie_agglomerative, AGG_GRID),
}

for name, sweep_df in threshold_sweeps.items():
    best = sweep_df.loc[sweep_df["mean_f1"].idxmax()]
    print(
        f"{name:18s} best mean_f1={best['mean_f1']:.4f} "
        f"@ threshold={best['distance_threshold']}  (mAP={best['mAP']:.4f})"
    )
    display(sweep_df)

process_movie_complete_linkage:   0%|          | 0/6 [00:00<?, ?it/s]

process_movie_establishing:   0%|          | 0/6 [00:00<?, ?it/s]

process_movie_agglomerative:   0%|          | 0/5 [00:00<?, ?it/s]

complete_linkage   best mean_f1=0.3674 @ threshold=0.8  (mAP=0.2809)


,distance_threshold,mean_f1,mAP
0,0.7,0.346728,0.280893
1,0.8,0.367384,0.280893
2,0.9,0.360805,0.280893
3,1.0,0.276720,0.280893
4,1.1,0.068276,0.280893
5,1.2,0.032455,0.280893


establishing       best mean_f1=0.3403 @ threshold=0.9  (mAP=0.3012)


,distance_threshold,mean_f1,mAP
0,0.7,0.311207,0.301234
1,0.8,0.334994,0.301234
2,0.9,0.340328,0.301234
3,1.0,0.335538,0.301234
4,1.1,0.331466,0.301234
5,1.2,0.302990,0.301234


agglomerative      best mean_f1=0.3515 @ threshold=0.5  (mAP=0.2899)


,distance_threshold,mean_f1,mAP
0,0.3,0.287078,0.289909
1,0.4,0.319468,0.289909
2,0.5,0.351523,0.289909
3,0.6,0.292521,0.289909
4,0.7,0.020162,0.289909


In [ ]:
def _sweep_boost(boosts, distance_threshold=1.0, tol_frames=TOL_FRAMES):
    rows = []
    for boost in tqdm(boosts, desc="establishing_boost"):
        f1s, aps = [], []
        for movie in bbc_movies:
            ev = evaluate_movie(
                movie,
                process_movie_establishing(
                    movie, distance_threshold=distance_threshold, establishing_boost=boost
                ),
                tol_frames,
            )
            f1s.append(ev["f1"])
            aps.append(ev["ap"])
        rows.append(
            {
                "establishing_boost": boost,
                "mean_f1": float(np.mean(f1s)),
                "mAP": float(np.mean(aps)),
            }
        )
    return pd.DataFrame(rows)


boost_sweep = _sweep_boost([0.0, 0.5, 1.0, 2.0, 3.0, 5.0])
print("establishing_boost sweep (boost=0 reproduces complete_linkage):")
display(boost_sweep)


def establishing_detector_report(tol_frames=TOL_FRAMES):
    """ShotVL is_establishing vs. actual GT scene starts, pooled over movies."""
    tp = fp = fn = 0
    for movie in bbc_movies:
        clean_starts = np.asarray(_clean_ranges_for_movie(movie))[:, 0]
        establishing = compute_establishing_labels(movie).astype(bool)
        gt = bbc_scenes_gt[bbc_scenes_gt["movie_name"] == movie].dropna(subset=["start_frame"])
        gt_starts = np.sort(gt["start_frame"].to_numpy().astype(int))
        true_start = np.array(
            [bool(np.any(np.abs(cs - gt_starts) <= tol_frames)) for cs in clean_starts]
        )
        tp += int(np.sum(establishing & true_start))
        fp += int(np.sum(establishing & ~true_start))
        fn += int(np.sum(~establishing & true_start))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


print(f"ShotVL establishing detector as a scene-start predictor (pooled, tol=+/-{TOL_FRAMES}):")
print(establishing_detector_report())

establishing_boost:   0%|          | 0/6 [00:00<?, ?it/s]

establishing_boost sweep (boost=0 reproduces complete_linkage):


,establishing_boost,mean_f1,mAP
0,0.0,0.276720,0.280893
1,0.5,0.339799,0.304455
2,1.0,0.335538,0.301234
3,2.0,0.297360,0.286325
4,3.0,0.261449,0.277398
5,5.0,0.249543,0.276324


ShotVL establishing detector as a scene-start predictor (pooled, tol=+/-2):
{'tp': 379, 'fp': 2089, 'fn': 258, 'precision': 0.15356564019448946, 'recall': 0.5949764521193093, 'f1': 0.244122383252818}


# Conclusion

Results below are frame-tolerant boundary metrics at **tol = +/-2 frames**, macro-averaged
over all 11 BBC movies (GT averages ~61 scenes/movie).

| method | mAP | mean-F1 | precision | recall | pred scenes (avg) |
|---|---|---|---|---|---|
| complete_linkage | 0.281 | **0.367** | 0.278 | 0.555 | 126 |
| establishing (boost 1.0) | **0.301** | 0.335 | 0.226 | 0.671 | 188 |
| agglomerative | 0.290 | 0.352 | 0.277 | 0.508 | 120 |

## What the numbers say


**1. Threshold tuning gave only marginal gains** (mAP is threshold-free, so flat within
each method):
- `complete_linkage`: already optimal at **0.8** -> F1 0.367; collapses above 1.0 (0.9->0.36, 1.1->0.07).
- `establishing`: best at **0.9** -> F1 0.340.
- `agglomerative`: best at **0.5** -> F1 0.352.

**2. The establishing prior helps *ranking* (AP), not *segmentation* (F1).**
The boost at threshold 1.0 shows a sweet spot: **boost 0.5 -> F1 0.340,
mAP 0.305**, both above no-boost (0.277 / 0.281); beyond that it degrades  The strong prior used was harmful, a light one is correct.

**3. The detector is noisy.** ShotVL establishing-vs-GT-scene-start:
**precision 0.15, recall 0.60** (tp 379, fp 2089, fn 258). It catches ~60% of real scene starts but flags 5x as many false ones; it over-segments.

## Bottom line
- **Best deployed segmentation (F1): `adjacent clustering with shot transition prior` @ threshold 0.8 (0.367).** The
  establishing prior does not beat it on F1 because the detector's 15% precision injects
  too many spurious cuts.
- **Best boundary ranking (AP): a *light* establishing prior (boost ~0.5).** Worth keeping only as a weak signal, not a hard cut.
- **`agglomerative`** lands in the middle on both — no clear win.
